In [ ]:
import os
import torch
import struct
import functools
import numpy as np
from typing import Tuple

RAND_SEED = 2026
DTYPES = [torch.float16, torch.bfloat16]
SHAPES = [(128, 64), (128, 128), (256, 64), (256, 128)]
PAD_SHAPES = [
    (90, 64),
    (150, 64),
    (128, 48),
    (128, 80),
    (150, 80),
    (90, 48),
    (90, 128),
    (150, 128),
    (150, 48),
    (90, 80),
]

FLOAT4_E2M1_MAX = 6.0
FLOAT8_E4M3_MAX = torch.finfo(torch.float8_e4m3fn).max

# E2M1 to float
# 0111 -> 6
# 0110 -> 4
# 0101 -> 3
# 0100 -> 2
# 0011 -> 1.5
# 0010 -> 1
# 0001 -> 0.5
# 0000 -> 0
E2M1_TO_FLOAT32 = [
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
    3.0,
    4.0,
    6.0,
    -0.0, # 0.0?
    -0.5,
    -1.0,
    -1.5,
    -2.0,
    -3.0,
    -4.0,
    -6.0,
]
BLOCK_SIZE = 16

def cuda_timeit(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        result = func(*args, **kwargs)
        end_event.record()

        torch.cuda.synchronize()  # 等待所有流完成
        elapsed_ms = start_event.elapsed_time(end_event)
        print(f"{func.__name__} 执行耗时: {elapsed_ms:.2f} ms")
        return result
    return wrapper


def cast_from_fp4(x, m, n):
    # The fp4 values are packed in uint8 as [v_1st | v_2nd]
    v_2nd = x & 0xF
    v_1st = (x >> 4) & 0xF
    c = torch.stack((v_2nd, v_1st), dim=-1)
    out = torch.tensor([E2M1_TO_FLOAT32[x] for x in c.flatten()])
    out = out.reshape(m, n).to(torch.float32)
    return out


def cast_to_fp4(x):
    sign = torch.sign(x)
    x = torch.abs(x)
    x[(x >= 0.0) & (x <= 0.25)] = 0.0
    x[(x > 0.25) & (x < 0.75)] = 0.5
    x[(x >= 0.75) & (x <= 1.25)] = 1.0
    x[(x > 1.25) & (x < 1.75)] = 1.5
    x[(x >= 1.75) & (x <= 2.5)] = 2.0
    x[(x > 2.5) & (x < 3.5)] = 3.0
    x[(x >= 3.5) & (x <= 5.0)] = 4.0
    x[x > 5.0] = 6.0
    return x * sign

def get_reciprocal(x):
    if isinstance(x, torch.Tensor):
        return torch.where(x == 0, torch.tensor(0.0, dtype=x.dtype), 1.0 / x)
    elif isinstance(x, (float, int)):
        # return 0.0 if x == 0 else 1.0 / x
        return (1 / 1e-8) if x == 0 else 1.0 / x
    else:
        raise TypeError("Input must be a float, int, or a torch.Tensor.")


def ref_nvfp4_quant(x, global_scale):
    assert global_scale.dtype == torch.float32
    assert x.ndim == 2
    m, n = x.shape
    x = torch.reshape(x, (m, n // BLOCK_SIZE, BLOCK_SIZE))
    vec_max = torch.max(torch.abs(x), dim=-1, keepdim=True)[0].to(torch.float32)
    scale = global_scale * (vec_max * get_reciprocal(FLOAT4_E2M1_MAX))
    # 从计算逻辑来看，scale 应该不会超出448, scale = FLOAT8_E4M3_MAX * vec_max / vall_max
    # scale = torch.clamp(scale, -FLOAT8_E4M3_MAX, FLOAT8_E4M3_MAX)
    scale = scale.to(torch.float8_e4m3fn).to(torch.float32)  # 超出448了，导致nan
    output_scale = get_reciprocal(scale * get_reciprocal(global_scale))

    scaled_x = x.to(torch.float32) * output_scale
    clipped_x = torch.clamp(scaled_x, -FLOAT4_E2M1_MAX, FLOAT4_E2M1_MAX).reshape(m, n)
    return cast_to_fp4(clipped_x), scale.squeeze(-1)


def recover_swizzled_scales(scale, m, k):
    '''
      block-wise FP4 量化的 scale 转换
      1.每个 scale 对应 16 列(CVT_FP4_SF_VEC_SIZE = 16)
      2.每 16 列共享一个 scale
      3.行方向每 128 行(32x4)为一个 tile
      4.列方向每 64 列(16x4)为一个 tile
    '''
    # m_tiles = (m + 128 - 1) // 128
    # f = BLOCK_SIZE * 4
    # k_tiles = (k + f - 1) // f
    # tmp = torch.reshape(a_sf_swizzled, (1, m_tiles, k_tiles, 32, 4, 4))
    # tmp = torch.permute(tmp, (0, 1, 4, 3, 2, 5))
    # out = tmp.reshape(m_tiles * 128, k_tiles * f // BLOCK_SIZE)
    # return out[0:m, 0:k]

    rounded_m = ((m + 128 - 1) // 128) * 128
    scale_k = k // BLOCK_SIZE
    rounded_k = ((scale_k + 4 - 1) // 4) * 4
    # Recover the swizzled scaling factor to linear layout
    tmp = torch.reshape(scale, (1, rounded_m // 128, rounded_k // 4, 32, 4, 4))
    tmp = torch.permute(tmp, (0, 1, 4, 3, 2, 5))
    result = torch.reshape(tmp, (rounded_m, rounded_k)).to(torch.float32)
    return result[:m, :rounded_k]


def convert_swizzled_to_linear(a_sf_swizzled, m, k):
    '''
      block-wise FP4 量化的 scale 转换
      1.每个 scale 对应 16 列(CVT_FP4_SF_VEC_SIZE = 16)
      2.每 16 列共享一个 scale
      3.行方向每 128 行(32x4)为一个 tile
      4.列方向每 64 列(16x4)为一个 tile
    '''
    m_tiles = (m + 128 - 1) // 128
    f = BLOCK_SIZE * 4
    k_tiles = (k + f - 1) // f
    
    # 计算所需的 rounded 尺寸
    rounded_m = m_tiles * 128
    rounded_n = k_tiles * f // BLOCK_SIZE
    
    # 检查输入是否需要padding
    if a_sf_swizzled.shape != (rounded_m, rounded_n):
        # 对输入进行padding到合适的尺寸
        padded_input = torch.zeros((rounded_m, rounded_n), 
                                   dtype=a_sf_swizzled.dtype, 
                                   device=a_sf_swizzled.device)
        # 复制有效数据
        actual_m = min(a_sf_swizzled.shape[0], rounded_m)
        actual_n = min(a_sf_swizzled.shape[1], rounded_n)
        padded_input[:actual_m, :actual_n] = a_sf_swizzled[:actual_m, :actual_n]
        a_sf_swizzled = padded_input
    
    # 按照原来的逻辑进行转换
    tmp = torch.reshape(a_sf_swizzled, (1, m_tiles, k_tiles, 32, 4, 4))
    tmp = torch.permute(tmp, (0, 1, 4, 3, 2, 5))
    out = tmp.reshape(m_tiles * 128, k_tiles * f // BLOCK_SIZE)
    
    # 输出时裁剪到指定的m,k大小
    return out[0:m, 0:k]

def convert_linear_to_swizzled(a_sf_linear, m, n):
    scale_n = n // BLOCK_SIZE
    rounded_m = ((m + 128 - 1) // 128) * 128
    rounded_n = ((scale_n + 4 - 1) // 4) * 4
    
    if a_sf_linear.shape != (rounded_m, rounded_n):
        padded = torch.zeros((rounded_m, rounded_n), dtype=a_sf_linear.dtype, device=a_sf_linear.device)
        padded[:m, :scale_n] = a_sf_linear[:m, :scale_n] if len(a_sf_linear.shape) == 2 else a_sf_linear
    else:
        padded = a_sf_linear
    
    tmp = torch.reshape(padded, (1, rounded_m // 128, 4, 32, rounded_n // 4, 4))
    tmp = torch.permute(tmp, (0, 1, 4, 3, 2, 5))
    
    result = torch.reshape(tmp, (rounded_m, rounded_n))
    return result

def ref_scaled_fp4_quant(x, global_scale):
    assert global_scale.dtype == torch.float32
    assert x.ndim == 2
    m, n = x.shape
    out_x, scale_linear = ref_nvfp4_quant(x, global_scale)
    scale_swizzled = convert_linear_to_swizzled(scale_linear, m, n)
    return out_x, scale_swizzled.to(torch.float8_e4m3fn)


def test_quantize_to_fp4(
    dtype: torch.dtype,
    shape: tuple[int, int],
) -> None:
    torch.manual_seed(RAND_SEED)
    torch.set_default_device("cuda:0")

    m, n = shape

    x = torch.randn((m, n), dtype=dtype)
    tensor_amax = torch.abs(x).max().to(torch.float32)
    global_scale = (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / tensor_amax
    out_ref, scale_linear = ref_nvfp4_quant(x, global_scale)
    print(out_ref.shape, scale_linear.shape)

    scale_swizzle = convert_linear_to_swizzled(scale_linear, m, n)
    scale_linear_check = recover_swizzled_scales(scale_swizzle, m, n)
    scale_linear_pad = convert_swizzled_to_linear(scale_swizzle, m, n)
    print(scale_swizzle.shape, scale_linear_check.shape, scale_linear_pad.shape)
    
    torch.testing.assert_close(scale_linear, scale_linear_check)
    torch.testing.assert_close(scale_linear, scale_linear_pad[:scale_linear.shape[0], :scale_linear.shape[1]])

@cuda_timeit
def break_fp4_bytes(a):
    assert a.dtype == torch.uint8
    m, n = a.shape
    a = a.flatten()
    # Get upper 4 bits
    highHalfByte = (a & 0xF0) >> 4
    # Get lower 4 bits
    lowHalfByte = a & 0x0F
    fH = torch.tensor([E2M1_TO_FLOAT32[x] for x in highHalfByte]).to(a.device)
    fL = torch.tensor([E2M1_TO_FLOAT32[x] for x in lowHalfByte]).to(a.device)
    # [0xAB, 0xCD] -> [0xB, 0xA, 0xD, 0xC], 0xCDAB
    out = torch.stack((fL, fH), dim=-1).reshape(m, n * 2)
    return out

@cuda_timeit
def break_fp4_bytes_v2(a):
    assert a.dtype == torch.uint8
    m, n = a.shape

    lut = torch.tensor(E2M1_TO_FLOAT32, dtype=torch.float32, device=a.device)
    
    # 提取高4位和低4位（注意：a >> 4 已得到高4位数值，无需再 & 0x0F）
    low = a & 0x0F               # 低4位，范围 0-15
    high = a >> 4                # 高4位，范围 0-15

    fL = lut[low.long()]         # [m, n]
    fH = lut[high.long()]        # [m, n]
    
    out = torch.stack((fL, fH), dim=-1).reshape(m, n * 2)
    return out

def pack_fp4_bytes_v1(a):
    assert a.dtype == torch.float32
    m, n2 = a.shape
    assert n2 % 2 == 0
    n = n2 // 2
    
    # [m, n*2] -> [m*n, 2]
    a = a.reshape(-1, 2)
    _E2M1_VALUES = torch.tensor(E2M1_TO_FLOAT32, dtype=torch.float32, device=a.device)
    
    # low 和 high 分别量化
    low = a[:, 0].unsqueeze(-1)   # [m*n, 1]
    high = a[:, 1].unsqueeze(-1)  # [m*n, 1]
    
    low_codes = torch.argmin(torch.abs(low - _E2M1_VALUES), dim=-1).to(torch.uint8)
    high_codes = torch.argmin(torch.abs(high - _E2M1_VALUES), dim=-1).to(torch.uint8)
    
    packed = (high_codes << 4) | low_codes
    return packed.reshape(m, n)

def pack_fp4_bytes(a):
    assert a.dtype == torch.float32
    m, n2 = a.shape
    assert n2 % 2 == 0
    n = n2 // 2
    
    # [m, n*2] -> [m*n, 2]
    a = a.reshape(-1, 2)
    
    low = a[:, 0]  # [m*n]
    high = a[:, 1] # [m*n]
    
    # 预计算所有可能的值
    values_list = E2M1_TO_FLOAT32.copy()
    values_list[8] = -0.0
    
    def quantize_channel(values):
        codes = torch.zeros(values.shape[0], dtype=torch.uint8, device=values.device)
        
        # 检查负零（索引8）
        neg_zero_mask = torch.isclose(values, torch.tensor(0.0, device=values.device)) & \
                        torch.signbit(values)
        codes[neg_zero_mask] = 8
        
        # 检查正零（索引0）
        pos_zero_mask = torch.isclose(values, torch.tensor(0.0, device=values.device)) & \
                       ~torch.signbit(values)
        codes[pos_zero_mask] = 0
        
        other_mask = ~(pos_zero_mask | neg_zero_mask)
        if other_mask.any():
            other_values = values[other_mask]
            for i, target_val in enumerate(values_list):
                if i == 0 or i == 8:  # 跳过两个零
                    continue
                mask_i = torch.isclose(other_values, torch.tensor(target_val, device=values.device))
                if mask_i.any():
                    # 找到这些值在原始数组中的索引
                    orig_indices = torch.nonzero(other_mask, as_tuple=True)[0]
                    indices_to_set = orig_indices[mask_i]
                    codes[indices_to_set] = i
        
        return codes
    
    low_codes = quantize_channel(low)
    high_codes = quantize_channel(high)
    
    packed = (high_codes << 4) | low_codes
    return packed.reshape(m, n)


def test_fp4_pack():
    torch.manual_seed(RAND_SEED)
    torch.set_default_device("cuda:0")
    # test_uint8 = torch.tensor([[0xAB, 0xCD], [0x12, 0x34]], dtype=torch.uint8)
    test_uint8 = torch.randint(0, 256, (20, 2)).to(torch.uint8)
    print(f"Original uint8:\n{test_uint8}")
    print(f"Shape: {test_uint8.shape}")
    
    # 正向
    broken = break_fp4_bytes_v2(test_uint8)
    print(f"\nBroken float32:\n{broken}")
    print(f"Shape: {broken.shape}")
    
    # 反向
    packed = pack_fp4_bytes(broken)
    print(f"\nPacked back:\n{packed}")
    print(f"Shape: {packed.shape}")
    
    # 验证
    print(f"\nMatch: {torch.equal(test_uint8, packed)}")

    mismatch_mask = test_uint8 != packed
    if mismatch_mask.any():
        print(f"Mismatches at indices: {torch.nonzero(mismatch_mask)}")
        print(f"Original values: {test_uint8[mismatch_mask]}")
        print(f"Packed values: {packed[mismatch_mask]}")


def dequantize_to_dtype(
    tensor_fp4, tensor_sf, global_scale, dtype, device, block_size=16
):
    """Dequantize the fp4 tensor back to high precision."""
    # Two fp4 values are packed into one uint8.
    assert tensor_fp4.dtype == torch.uint8
    m, packed_k = tensor_fp4.shape
    k = packed_k * 2
    tensor_f32 = break_fp4_bytes_v2(tensor_fp4)
    tensor_f32 = tensor_f32.reshape(m, k // block_size, block_size)
    tensor_sf = tensor_sf.view(torch.float8_e4m3fn)
    tensor_sf_linear = convert_swizzled_to_linear(tensor_sf, m, k)
    block_scale = tensor_sf_linear.to(torch.float32) / global_scale

    # scale the tensor
    print(tensor_f32.shape, block_scale.shape)
    out = (tensor_f32 * block_scale.unsqueeze(-1)).reshape(m, k)
    return out


def get_ref_nvfp4_mul_results(
    a_fp4,
    b_fp4,
    a_sf,
    b_sf,
    a_global_scale,
    b_global_scale,
    m,
    n,
    dtype,
    block_size,
    device,
):
    _, m_k = a_fp4.shape
    _, n_k = b_fp4.shape
    assert m_k == n_k
    a_in_dtype = dequantize_to_dtype(
        a_fp4, a_sf, a_global_scale, dtype=dtype, device=device, block_size=block_size
    )
    b_in_dtype = dequantize_to_dtype(
        b_fp4, b_sf, b_global_scale, dtype=dtype, device=device, block_size=block_size
    )
    return torch.matmul(a_in_dtype, b_in_dtype.t())

def get_ref_fp16_mul_results(
    a_half,
    b_half):
    _, m_k = a_half.shape
    _, n_k = b_half.shape
    assert m_k == n_k
    return torch.matmul(a_half, b_half.t())


def test_nvfp4_gemm(
    dtype: torch.dtype,
    shape: tuple[int, int],
    test_mode: bool = False,
) -> None:
    torch.manual_seed(RAND_SEED)
    torch.set_default_device("cuda:0")
    m, n, packed_k = shape
    k = packed_k * 2
    block_size = BLOCK_SIZE
    a_dtype = torch.randn((m, k), dtype=dtype, device="cuda")
    b_dtype = torch.randn((n, k), dtype=dtype, device="cuda")

    a_global_scale = (
        (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / torch.amax(torch.abs(a_dtype.flatten()), dim=-1).to(torch.float32)
    ).to(torch.float32)
    b_global_scale = (
        (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / torch.amax(torch.abs(b_dtype.flatten()), dim=-1).to(torch.float32)
    ).to(torch.float32)
    alpha = 1.0 / (a_global_scale * b_global_scale)
    
    if test_mode:
        a_fp4_float, a_scale_linear = ref_nvfp4_quant(a_dtype, a_global_scale)
        a_scale_swizzled = convert_linear_to_swizzled(a_scale_linear, m, k)
        a_scale_interleaved = a_scale_swizzled.to(torch.float8_e4m3fn)
        a_fp4 = pack_fp4_bytes(a_fp4_float)

        b_fp4_float, b_scale_linear = ref_nvfp4_quant(b_dtype, b_global_scale)
        b_scale_swizzled = convert_linear_to_swizzled(b_scale_linear, n, k)
        b_scale_interleaved = b_scale_swizzled.to(torch.float8_e4m3fn)
        b_fp4 = pack_fp4_bytes(b_fp4_float)
        print(a_scale_swizzled.shape, b_scale_swizzled.shape)
        print(a_global_scale, b_global_scale)
        print(a_dtype)
        print(b_dtype)
        # print(a_scale_swizzled)
        # print(b_scale_swizzled)
        print(a_scale_linear)
        print(b_scale_linear)
        print(a_fp4_float)
        print(b_fp4_float)
    else:
        a_fp4_float, a_scale_interleaved = ref_scaled_fp4_quant(a_dtype, a_global_scale)
        b_fp4_float, b_scale_interleaved = ref_scaled_fp4_quant(b_dtype, b_global_scale)
        a_fp4 = pack_fp4_bytes(a_fp4_float)
        b_fp4 = pack_fp4_bytes(b_fp4_float)

    expected_out = get_ref_nvfp4_mul_results(
        a_fp4,
        b_fp4,
        a_scale_interleaved,
        b_scale_interleaved,
        a_global_scale,
        b_global_scale,
        m,
        n,
        dtype,
        block_size,
        "cuda",
    )
    print(expected_out, expected_out.shape)

    half_out = get_ref_fp16_mul_results(a_dtype, b_dtype)
    print(half_out, half_out.shape)
    # print(expected_out[95,214], half_out[95,214])
    torch.testing.assert_close(half_out, expected_out.to(dtype=dtype), atol=1e-1, rtol=1e-1)


class NVFP4Data:
    def __init__(self):
        self.M = 0
        self.N = 0
        self.K = 0
        self.A_fp16 = None
        self.B_fp16 = None
        self.A_fp4 = None
        self.B_fp4 = None
        self.A_scales = None
        self.B_scales = None
        self.output = None
        self.output_ref = None
        self.a_global_scale = 0.0
        self.b_global_scale = 0.0
        self.alpha = 0.0
    
    def save(self, filename: str):
        """保存数据到二进制文件"""
        with open(filename, 'wb') as f:
            # 写入维度信息
            f.write(struct.pack('iii', self.M, self.N, self.K))
            
            # 写入全局缩放因子
            f.write(struct.pack('fff', 
                              float(self.a_global_scale), 
                              float(self.b_global_scale), 
                              float(self.alpha)))
            
            # 写入数据大小
            size_A_fp16 = self.A_fp16.numel() if self.A_fp16 is not None else 0
            size_B_fp16 = self.B_fp16.numel() if self.B_fp16 is not None else 0
            size_A_fp4 = self.A_fp4.numel() if self.A_fp4 is not None else 0
            size_B_fp4 = self.B_fp4.numel() if self.B_fp4 is not None else 0
            size_A_scales = self.A_scales.numel() if self.A_scales is not None else 0
            size_B_scales = self.B_scales.numel() if self.B_scales is not None else 0
            size_output = self.output.numel() if self.output is not None else 0
            size_output_ref = self.output_ref.numel() if self.output_ref is not None else 0
            
            f.write(struct.pack('QQQQQQQQ', 
                              size_A_fp16, size_B_fp16,
                              size_A_fp4, size_B_fp4,
                              size_A_scales, size_B_scales,
                              size_output, size_output_ref))
            
            # 写入数据
            if self.A_fp16 is not None:
                f.write(self.A_fp16.cpu().numpy().tobytes())
            if self.B_fp16 is not None:
                f.write(self.B_fp16.cpu().numpy().tobytes())
            if self.A_fp4 is not None:
                f.write(self.A_fp4.cpu().numpy().tobytes())
            if self.B_fp4 is not None:
                f.write(self.B_fp4.cpu().numpy().tobytes())
            if self.A_scales is not None:
                f.write(self.A_scales.cpu().numpy().tobytes())
            if self.B_scales is not None:
                f.write(self.B_scales.cpu().numpy().tobytes())
            if self.output is not None:
                f.write(self.output.cpu().numpy().tobytes())
            if self.output_ref is not None:
                f.write(self.output_ref.cpu().numpy().tobytes())
        
        print(f"Data saved to {filename}")
    
    @classmethod
    def load(cls, filename: str, device="cuda", load_from_cpp=False):
        """从二进制文件加载数据"""
        data = cls()
        
        with open(filename, 'rb') as f:
            # 读取维度信息
            data.M, data.N, data.K = struct.unpack('iii', f.read(12))
            
            # 读取全局缩放因子
            a_gs, b_gs, alpha = struct.unpack('fff', f.read(12))
            data.a_global_scale = torch.tensor(a_gs, dtype=torch.float32)
            data.b_global_scale = torch.tensor(b_gs, dtype=torch.float32)
            data.alpha = torch.tensor(alpha, dtype=torch.float32)
            
            # 读取数据大小
            sizes = struct.unpack('QQQQQQQQ', f.read(64))
            (size_A_fp16, size_B_fp16, 
             size_A_fp4, size_B_fp4,
             size_A_scales, size_B_scales,
             size_output, size_output_ref) = sizes
            print(f"Sizes: {sizes}")
            
            def get_pad_dim(m, n):
                scale_n = n // BLOCK_SIZE
                rounded_m = ((m + 128 - 1) // 128) * 128
                rounded_n = ((scale_n + 4 - 1) // 4) * 4
                return (rounded_m, rounded_n)
            
            # 读取数据
            if size_A_fp16 > 0:
                buffer = f.read(size_A_fp16 * 2)  # FP16是2字节
                arr = np.frombuffer(buffer, dtype=np.float16).reshape(data.M, data.K)
                data.A_fp16 = torch.from_numpy(arr.copy()).to(device=device, dtype=torch.float16)
            
            if size_B_fp16 > 0:
                buffer = f.read(size_B_fp16 * 2)
                arr = np.frombuffer(buffer, dtype=np.float16).reshape(data.N, data.K)
                data.B_fp16 = torch.from_numpy(arr.copy()).to(device=device, dtype=torch.float16)
            
            if size_A_fp4 > 0:
                buffer = f.read(size_A_fp4 * 8)  # int64_t是8字节
                arr = np.frombuffer(buffer, dtype=np.uint8)
                data.A_fp4 = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.uint8), (data.M, data.K // 2))
            
            if size_B_fp4 > 0:
                buffer = f.read(size_B_fp4 * 8)
                arr = np.frombuffer(buffer, dtype=np.uint8)
                data.B_fp4 = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.uint8), (data.N, data.K // 2))
            
            if size_A_scales > 0:
                if not load_from_cpp:
                    buffer = f.read(size_A_scales * 4)
                    arr = np.frombuffer(buffer, dtype=np.int32)
                    data.A_scales = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.int32).to(torch.float8_e4m3fn), get_pad_dim(data.M, data.K))
                else:
                    buffer = f.read(size_A_scales * 4)
                    arr = np.frombuffer(buffer, dtype=np.uint8)
                    data.A_scales = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.uint8).view(torch.float8_e4m3fn), get_pad_dim(data.M, data.K))
            if size_B_scales > 0:
                if not load_from_cpp:
                    buffer = f.read(size_B_scales * 4)
                    arr = np.frombuffer(buffer, dtype=np.int32)
                    data.B_scales = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.int32).to(torch.float8_e4m3fn), get_pad_dim(data.N, data.K))
                else:
                    buffer = f.read(size_B_scales * 4)
                    arr = np.frombuffer(buffer, dtype=np.uint8)
                    data.B_scales = torch.reshape(torch.from_numpy(arr.copy()).to(device=device, dtype=torch.uint8).view(torch.float8_e4m3fn), get_pad_dim(data.N, data.K))
            if size_output > 0:
                buffer = f.read(size_output * 2)
                arr = np.frombuffer(buffer, dtype=np.float16).reshape(data.M, data.N)
                data.output = torch.from_numpy(arr.copy()).to(device=device, dtype=torch.float16)

            if size_output_ref > 0:
                buffer = f.read(size_output_ref * 2)
                arr = np.frombuffer(buffer, dtype=np.float16).reshape(data.M, data.N)
                data.output_ref = torch.from_numpy(arr.copy()).to(device=device, dtype=torch.float16)

        print(f"Data loaded from {filename}")
        print(f"M={data.M}, N={data.N}, K={data.K}")
        return data


def generate_and_save_test_data(
    M: int, N: int, K: int,
    output_file: str,
    dtype: torch.dtype = torch.float16):
    
    torch.manual_seed(RAND_SEED)
    device = "cuda"
    
    # 生成随机数据
    a_dtype = torch.randn((M, K), dtype=dtype, device=device)
    b_dtype = torch.randn((N, K), dtype=dtype, device=device)
    
    # 计算全局缩放因子
    a_global_scale = ((FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / 
                     torch.amax(torch.abs(a_dtype.flatten()), dim=-1).to(torch.float32)).to(torch.float32)
    b_global_scale = ((FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / 
                     torch.amax(torch.abs(b_dtype.flatten()), dim=-1).to(torch.float32)).to(torch.float32)
    alpha = 1.0 / (a_global_scale * b_global_scale)
    
    # 量化
    a_fp4_float, a_scale_interleaved = ref_scaled_fp4_quant(a_dtype, a_global_scale)
    b_fp4_float, b_scale_interleaved = ref_scaled_fp4_quant(b_dtype, b_global_scale)
    
    a_fp4 = pack_fp4_bytes(a_fp4_float)
    b_fp4 = pack_fp4_bytes(b_fp4_float)
    
    # 保存数据
    data = NVFP4Data()
    data.M = M
    data.N = N
    data.K = K
    data.A_fp16 = a_dtype
    data.B_fp16 = b_dtype
    data.A_fp4 = torch.from_numpy(np.frombuffer(
        a_fp4.detach().cpu().numpy().tobytes(), dtype=np.int64).copy()).to(device)
    data.B_fp4 = torch.from_numpy(np.frombuffer(
        b_fp4.detach().cpu().numpy().tobytes(), dtype=np.int64).copy()).to(device)
    print("fp4 shape: ", a_fp4.shape, b_fp4.shape)
    print("scale shape: ", a_scale_interleaved.shape, b_scale_interleaved.shape)
    data.A_scales = a_scale_interleaved.to(torch.int32)
    data.B_scales = b_scale_interleaved.to(torch.int32)
    data.a_global_scale = a_global_scale
    data.b_global_scale = b_global_scale
    data.alpha = alpha.detach().clone().to(dtype=torch.float32, device=device)
    
    # 计算参考输出
    data.output = get_ref_nvfp4_mul_results(
        a_fp4, b_fp4,
        a_scale_interleaved, b_scale_interleaved,
        a_global_scale, b_global_scale,
        M, N, dtype, BLOCK_SIZE, device).to(dtype=torch.float16, device=device)
    data.output_half = get_ref_fp16_mul_results(a_dtype, b_dtype)
    # print(a_dtype)
    # print(b_dtype)
    # print(data.output)
    
    data.save(output_file)
    return data

def compare_results(cpp_output_file: str, python_data_file: str, rtol: float = 1e-2, atol: float = 1e-2, device="cuda"):
    cpp_data = NVFP4Data.load(cpp_output_file, device, load_from_cpp=True)
    python_data = NVFP4Data.load(python_data_file, device)

    assert cpp_data.M == python_data.M
    assert cpp_data.N == python_data.N
    assert cpp_data.K == python_data.K

    torch.testing.assert_close(python_data.A_fp4, cpp_data.A_fp4)
    torch.testing.assert_close(python_data.B_fp4, cpp_data.B_fp4)
    torch.testing.assert_close(python_data.A_scales, cpp_data.A_scales)
    torch.testing.assert_close(python_data.B_scales, cpp_data.B_scales)
    torch.testing.assert_close(python_data.a_global_scale, cpp_data.a_global_scale)
    torch.testing.assert_close(python_data.b_global_scale, cpp_data.b_global_scale)
    torch.testing.assert_close(python_data.alpha, cpp_data.alpha)

    if cpp_data.output is not None and python_data.output is not None:
        cpp_output = cpp_data.output
        python_output = python_data.output
        
        abs_diff = torch.abs(cpp_output - python_output)
        rel_diff = abs_diff / (torch.abs(python_output) + 1e-8)
        
        max_abs_error = torch.max(abs_diff).item()
        max_rel_error = torch.max(rel_diff).item()
        mean_abs_error = torch.mean(abs_diff).item()
        mean_rel_error = torch.mean(rel_diff).item()
        
        print("\n=== 结果比较 ===")
        print(f"最大绝对误差: {max_abs_error:.6e}")
        print(f"最大相对误差: {max_rel_error:.6e}")
        print(f"平均绝对误差: {mean_abs_error:.6e}")
        print(f"平均相对误差: {mean_rel_error:.6e}")
        
        # 检查误差是否在容忍范围内
        abs_ok = torch.allclose(cpp_output, python_output, rtol=rtol, atol=atol)
        print(f"绝对误差检查: {'通过' if abs_ok else '失败'}")
        
        return max_abs_error, max_rel_error, abs_ok
    else:
        print("错误：输出数据为空")
        return 0, 0, False
    
def test_nvfp4_gemm_from_data(input_file):
    data = NVFP4Data.load(input_file)
    a_global_scale = ((FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / 
                     torch.amax(torch.abs(data.A_fp16.flatten()), dim=-1).to(torch.float32)).to(torch.float32)
    b_global_scale = ((FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / 
                     torch.amax(torch.abs(data.B_fp16.flatten()), dim=-1).to(torch.float32)).to(torch.float32)
    alpha = 1.0 / (a_global_scale * b_global_scale)
    a_fp4_float, a_scale_interleaved = ref_scaled_fp4_quant(data.A_fp16, data.a_global_scale)
    b_fp4_float, b_scale_interleaved = ref_scaled_fp4_quant(data.B_fp16, data.b_global_scale)
    a_fp4 = pack_fp4_bytes(a_fp4_float)
    b_fp4 = pack_fp4_bytes(b_fp4_float)

    expected_out = get_ref_nvfp4_mul_results(
        a_fp4,
        b_fp4,
        a_scale_interleaved,
        b_scale_interleaved,
        data.a_global_scale,
        data.b_global_scale,
        data.M,
        data.N,
        data.A_fp16.dtype,
        BLOCK_SIZE,
        "cuda",
    )

    torch.testing.assert_close(data.A_fp4, a_fp4)
    torch.testing.assert_close(data.B_fp4, b_fp4)
    torch.testing.assert_close(data.A_scales, a_scale_interleaved)
    torch.testing.assert_close(data.B_scales, b_scale_interleaved)
    torch.testing.assert_close(data.a_global_scale.to("cuda"), a_global_scale)
    torch.testing.assert_close(data.b_global_scale.to("cuda"), b_global_scale)
    torch.testing.assert_close(data.alpha.to("cuda"), alpha)

    print(data.M, data.N, data.K)
    # print(a_global_scale, b_global_scale, alpha)
    # print(data.a_global_scale, data.b_global_scale, data.alpha)
    # print(data.A_fp16)
    # print(data.B_fp16)
    print(expected_out, expected_out.shape)
    print(data.output, data.output.shape)
    torch.testing.assert_close(data.output, expected_out.to(dtype=data.A_fp16.dtype), atol=1e-1, rtol=1e-1)

def mopt_nvfp4_quant(x):
    from modelopt.torch.quantization.qtensor import NVFP4QTensor
    tensor_amax = torch.abs(x).max().to(torch.float32)
    global_scale_mopt =  tensor_amax / (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX)
    weight_scaling_factor, weight_scaling_factor_2 = (
        NVFP4QTensor.get_weights_scaling_factor(x, 16, global_scale_mopt))
    quantized_weight, _, _ = (
        NVFP4QTensor.quantize(x, 16, weight_scaling_factor, weight_scaling_factor_2, try_tensorrt=True))

    return quantized_weight._quantized_data, weight_scaling_factor, 1.0 / weight_scaling_factor_2


def _get_padded_shape(M, K):
    round_up_multiple = lambda x, m: (x + m - 1) // m * m
    M_padded = round_up_multiple(M, 128)
    K_padded = round_up_multiple(K, 4)
    return M_padded, K_padded

def _prepare_scales(scales: torch.Tensor, scale_ndim: int = 2) -> torch.Tensor:
    B, M, K = scales.shape
    M_padded, K_padded = _get_padded_shape(M, K)
    
    # Create padded tensor
    padded_scales = torch.zeros((B, M_padded, K_padded), dtype=scales.dtype, device=scales.device)
    padded_scales[:B, :M, :K] = scales
    
    batches, rows, cols = padded_scales.shape
    assert rows % 128 == 0
    assert cols % 4 == 0
    
    # Reshape and permute
    padded_scales = padded_scales.reshape(batches, rows // 128, 4, 32, cols // 4, 4)
    padded_scales = padded_scales.permute((0, 1, 4, 3, 2, 5))
    padded_scales = padded_scales.contiguous()
    
    # Final reshape
    if scale_ndim == 2:
        padded_scales = padded_scales.reshape(M_padded, K_padded)
    else:
        padded_scales = padded_scales.reshape(B, M_padded, K_padded)
    
    return padded_scales

def test_mopt_nvfp4_quant(
    dtype: torch.dtype,
    shape: tuple[int, int],
) -> None:
    from modelopt.torch.quantization.qtensor import NVFP4QTensor
    torch.manual_seed(RAND_SEED)
    torch.set_default_device("cuda:0")

    m, n = shape
    x = torch.randn((m, n), dtype=dtype)
    tensor_amax = torch.abs(x).max().to(torch.float32)
    global_scale = (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / tensor_amax
    out_ref, scale_linear = ref_nvfp4_quant(x, global_scale)
    packed_ref = pack_fp4_bytes(out_ref)
    scale_linear_ref = scale_linear.to(torch.float8_e4m3fn)
    print(packed_ref.shape, scale_linear_ref.shape)
    scale_swizzle = convert_linear_to_swizzled(scale_linear_ref, m, n)

    quantized_weight, weight_scaling_factor, weight_scaling_factor_2 = mopt_nvfp4_quant(x)
    qscales = _prepare_scales(weight_scaling_factor.unsqueeze(0), scale_ndim=3).squeeze(0)
    print(scale_swizzle.shape, qscales.shape)
    print(quantized_weight.shape, weight_scaling_factor.shape)
    print(packed_ref)
    print(quantized_weight)
    print(scale_linear_ref, weight_scaling_factor)
    print(global_scale, weight_scaling_factor_2)
    print(packed_ref[69, 246], quantized_weight[69, 246])
    print(packed_ref[2429, 950], quantized_weight[2429, 950])

    torch.testing.assert_close(global_scale, weight_scaling_factor_2)
    torch.testing.assert_close(scale_linear_ref, weight_scaling_factor)
    torch.testing.assert_close(scale_swizzle, qscales)
    torch.testing.assert_close(packed_ref, quantized_weight)

# test_mopt_nvfp4_quant(torch.float16, (1, 64))
test_mopt_nvfp4_quant(torch.float16, (6144, 2048))

# test_quantize_to_fp4(torch.float16, (1, 64))
# test_quantize_to_fp4(torch.float16, (6144, 2048))
# test_fp4_pack()
# test_nvfp4_gemm(torch.float16, (1, 1, 64//2), True)
# generate_and_save_test_data(1, 8, 64, "nvfp4_testdata_1_8_64.bin")
# test_nvfp4_gemm_from_data("nvfp4_testdata_1_8_64.bin")

# HOME_DIR = "/home/huye/code/opensources/ml/vlm/ml/vlm_fastv/TensorRT-Edge-LLM/tests_cpp/resources/moe/nvfp4"
# compare_results(os.path.join(HOME_DIR, "cpp_nvfp4_1_8_64.bin"), os.path.join(HOME_DIR, "nvfp4_testdata_1_8_64.bin"), device = "cpu")


In [ ]:
import torch

def get_test_data():
     input_A = torch.tensor(
          [[-1.8140e-01,  5.6543e-01, -7.6660e-01,  2.3340e-01,  1.3098e-01,
             1.3340e+00, -4.4922e-01,  3.4180e-01, -1.9760e-03, -2.5464e-01,
             2.0020e-01,  1.6484e+00,  1.0889e+00,  4.7241e-01, -1.0684e+00,
             1.9854e+00,  1.9590e+00,  1.0371e+00,  1.7412e+00, -1.0918e+00,
             5.9180e-01,  9.0186e-01, -2.8574e+00, -9.1431e-02,  2.6929e-01,
             -4.2212e-01, -2.2441e+00, -8.1738e-01,  1.1328e+00, -1.1631e+00,
             3.0615e-01, -4.0088e-01,  1.0156e+00,  2.2620e-01,  3.6279e-01,
             -2.4097e-01, -1.6836e+00,  1.7639e-01, -4.5850e-01, -5.2686e-01,
             -8.1055e-01, -6.9678e-01, -4.5068e-01, -2.8442e-01, -1.9219e+00,
             -7.6782e-02,  7.1582e-01,  7.4463e-01,  8.5498e-01,  6.3232e-01,
             8.0371e-01, -8.7463e-02,  7.2205e-02, -1.6904e+00,  6.0156e-01,
             -7.8125e-02, -7.8320e-01,  9.0918e-01, -1.2783e+00, -3.5938e-01,
             9.4043e-01, -9.6289e-01, -4.2065e-01, -9.1064e-01]], dtype=torch.float16)

     input_B = torch.tensor(
         [[ 1.2488e-01, -1.9093e-03,  3.3447e-01, -4.9927e-01, -1.8140e-01,
              -4.6875e-01, -2.5352e+00, -9.5703e-02,  1.2805e-01, -1.9180e+00,
               7.1338e-01, -3.4023e+00,  1.0713e+00, -2.0059e+00, -3.8940e-01,
              -1.3848e+00,  1.4389e-02, -9.2621e-03, -8.6328e-01, -2.1387e-01,
              -8.6279e-01,  6.5967e-01,  2.7783e-01, -5.4395e-01,  9.8486e-01,
              -6.9092e-02,  3.9014e-01, -1.3232e+00,  8.1726e-02, -2.1602e+00,
               9.1162e-01, -6.0205e-01,  4.1553e-01, -9.9805e-01, -5.2344e-01,
              -5.7617e-01,  1.2373e+00,  1.3262e+00, -6.1133e-01, -3.1958e-01,
              -8.2520e-02,  5.6519e-02, -6.3086e-01, -1.5918e-01,  2.7148e-01,
               7.9199e-01, -1.1309e+00,  6.3965e-01,  1.0195e+00,  1.6416e+00,
               3.8525e-01,  1.8691e+00, -3.2568e-01, -1.9580e-01, -2.2229e-01,
              -1.6113e-01, -1.0850e+00, -2.4414e-01,  1.0518e+00, -1.2041e+00,
               1.8281e+00, -6.7334e-01, -1.2067e-01,  1.7090e-01]], dtype=torch.float16)

     input_A_nvfp4 = torch.tensor(
         [[-0.5000,  1.5000, -2.0000,  0.5000,  0.5000,  4.0000, -1.5000,  1.0000,
         -0.0000, -0.5000,  0.5000,  4.0000,  3.0000,  1.5000, -3.0000,  6.0000,
          4.0000,  2.0000,  4.0000, -2.0000,  1.0000,  2.0000, -6.0000, -0.0000,
          0.5000, -1.0000, -4.0000, -1.5000,  2.0000, -2.0000,  0.5000, -1.0000,
          3.0000,  0.5000,  1.0000, -1.0000, -6.0000,  0.5000, -1.5000, -1.5000,
         -3.0000, -2.0000, -1.5000, -1.0000, -6.0000, -0.5000,  2.0000,  2.0000,
          3.0000,  2.0000,  3.0000, -0.5000,  0.5000, -6.0000,  2.0000, -0.5000,
         -3.0000,  3.0000, -4.0000, -1.5000,  3.0000, -4.0000, -1.5000, -3.0000]])
     input_A_scale = torch.tensor(
         [[320., 448., 288., 256.]])

     input_B_nvfp4 = torch.tensor(
         [[ 0.0000, -0.0000,  0.5000, -1.0000, -0.5000, -1.0000, -4.0000, -0.0000,
          0.0000, -3.0000,  1.5000, -6.0000,  2.0000, -4.0000, -0.5000, -2.0000,
          0.0000, -0.0000, -2.0000, -0.5000, -2.0000,  2.0000,  1.0000, -1.5000,
          3.0000, -0.0000,  1.0000, -4.0000,  0.0000, -6.0000,  3.0000, -1.5000,
          2.0000, -4.0000, -2.0000, -3.0000,  6.0000,  6.0000, -3.0000, -1.5000,
         -0.5000,  0.5000, -3.0000, -0.5000,  1.0000,  4.0000, -6.0000,  3.0000,
          3.0000,  6.0000,  1.5000,  6.0000, -1.0000, -0.5000, -0.5000, -0.5000,
         -4.0000, -1.0000,  3.0000, -4.0000,  6.0000, -2.0000, -0.5000,  0.5000]])
     input_B_scale = torch.tensor(
         [[448., 288., 176., 240.]])
     return input_A, input_B, input_A_nvfp4, input_A_scale, input_B_nvfp4, input_B_scale

def test_nvfp4_matmul():
    block_size = 16
    FLOAT4_E2M1_MAX = 6.0
    FLOAT8_E4M3_MAX  = torch.finfo(torch.float8_e4m3fn).max

    input_A, input_B, input_A_nvfp4, input_A_scale, input_B_nvfp4, input_B_scale = get_test_data()
    out_half = torch.matmul(input_A, input_B.t())

    a_global_scale = (
        (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / torch.amax(torch.abs(input_A.flatten()), dim=-1).to(torch.float32)
    ).to(torch.float32)
    b_global_scale = (
        (FLOAT8_E4M3_MAX * FLOAT4_E2M1_MAX) / torch.amax(torch.abs(input_B.flatten()), dim=-1).to(torch.float32)
    ).to(torch.float32)
    print(a_global_scale, b_global_scale)

    m, k = input_A.shape
    cvt_input_A = input_A_nvfp4.reshape(m, k // block_size, block_size)
    cvt_input_A_scale = input_A_scale.to(torch.float32) / a_global_scale
    cvt_A = (cvt_input_A * cvt_input_A_scale.unsqueeze(-1)).reshape(m, k)

    n, k = input_B.shape
    cvt_input_B = input_B_nvfp4.reshape(n, k // block_size, block_size)
    cvt_input_B_scale = input_B_scale.to(torch.float32) / b_global_scale
    cvt_B = (cvt_input_B * cvt_input_B_scale.unsqueeze(-1)).reshape(n, k)

    out_nvfp4 = torch.matmul(cvt_A, cvt_B.t())
    print(out_half)
    print(out_nvfp4)
    torch.testing.assert_close(out_half, out_nvfp4.to(dtype=torch.float16), atol=1e-1, rtol=1e-1)


test_nvfp4_matmul()


## moe nvfp4 data gen

In [ ]:
import os
import torch
import functools
import torch.nn as nn
import numpy as np
from safetensors.torch import save_file, load_file
from typing import Tuple

try:
    from modelopt.torch.quantization.qtensor import NVFP4QTensor
    MODELOPT_AVAILABLE = True
except ImportError:
    print("警告: modelopt未安装，将使用模拟的NVFP4量化（仅用于生成测试数据框架）")
    MODELOPT_AVAILABLE = False

# ============ 配置 ============
BLOCK_SIZE = 16  # NVFP4 块大小
# 0000 -> 0
E2M1_TO_FLOAT32 = [
    0.0,
    0.5,
    1.0,
    1.5,
    2.0,
    3.0,
    4.0,
    6.0,
    -0.0, # 0.0?
    -0.5,
    -1.0,
    -1.5,
    -2.0,
    -3.0,
    -4.0,
    -6.0,
]
DATA_ROOT_DIR = '/workspace/app/ml/vlm_fastv/TensorRT-Edge-LLM/tests_cpp/resources/moe'  # 请修改为实际路径

# ============ 辅助函数 ============
def cuda_timeit(func):
    @functools.wraps(func)
    def wrapper(*args, **kwargs):
        start_event = torch.cuda.Event(enable_timing=True)
        end_event = torch.cuda.Event(enable_timing=True)

        start_event.record()
        result = func(*args, **kwargs)
        end_event.record()

        torch.cuda.synchronize()  # 等待所有流完成
        elapsed_ms = start_event.elapsed_time(end_event)
        print(f"{func.__name__} 执行耗时: {elapsed_ms:.2f} ms")
        return result
    return wrapper


def _get_padded_shape(M: int, K: int) -> Tuple[int, int]:
    round_up_multiple = lambda x, m: (x + m - 1) // m * m
    M_padded = round_up_multiple(M, 128)
    K_padded = round_up_multiple(K, 4)
    return M_padded, K_padded

def _swizzle_scales(scales: torch.Tensor, scale_ndim: int = 2) -> torch.Tensor:
    B, M, K = scales.shape
    M_padded, K_padded = _get_padded_shape(M, K)

    padded_scales = torch.zeros((B, M_padded, K_padded), dtype=scales.dtype, device=scales.device)
    padded_scales[:B, :M, :K] = scales

    batches, rows, cols = padded_scales.shape
    assert rows % 128 == 0
    assert cols % 4 == 0

    # Reshape and permute
    padded_scales = padded_scales.reshape(batches, rows // 128, 4, 32, cols // 4, 4)
    padded_scales = padded_scales.permute((0, 1, 4, 3, 2, 5))
    padded_scales = padded_scales.contiguous()

    # Final reshape
    if scale_ndim == 2:
        padded_scales = padded_scales.reshape(M_padded, K_padded)
    else:
        padded_scales = padded_scales.reshape(B, M_padded, K_padded)

    return padded_scales

@cuda_timeit
def _break_fp4_bytes(a):
    assert a.dtype == torch.uint8
    m, n = a.shape
    a = a.flatten()
    # Get upper 4 bits
    highHalfByte = (a & 0xF0) >> 4
    # Get lower 4 bits
    lowHalfByte = a & 0x0F
    fH = torch.tensor([E2M1_TO_FLOAT32[x] for x in highHalfByte]).to(a.device)
    fL = torch.tensor([E2M1_TO_FLOAT32[x] for x in lowHalfByte]).to(a.device)
    # [0xAB, 0xCD] -> [0xB, 0xA, 0xD, 0xC], 0xCDAB
    out = torch.stack((fL, fH), dim=-1).reshape(m, n * 2)
    return out

@cuda_timeit
def _break_fp4_bytes_v2(a):
    assert a.dtype == torch.uint8
    m, n = a.shape

    lut = torch.tensor(E2M1_TO_FLOAT32, dtype=torch.float32, device=a.device)
    
    # 提取高4位和低4位（注意：a >> 4 已得到高4位数值，无需再 & 0x0F）
    low = a & 0x0F               # 低4位，范围 0-15
    high = a >> 4                # 高4位，范围 0-15

    fL = lut[low.long()]         # [m, n]
    fH = lut[high.long()]        # [m, n]
    
    out = torch.stack((fL, fH), dim=-1).reshape(m, n * 2)
    return out


def _quantize_to_nvfp4(
        data: torch.Tensor,
        preset_global_scale=None,
    ) -> Tuple[torch.Tensor, torch.Tensor]:
        tensor_amax = torch.abs(data).max().to(torch.float32)
        global_scale_mopt =  tensor_amax / (448.0 * 6.0)
        if preset_global_scale is not None:
            global_scale_mopt = 1.0 / preset_global_scale
        qscales_linear, scaling_factor_2 = (
            NVFP4QTensor.get_weights_scaling_factor(data, BLOCK_SIZE, global_scale_mopt))
        quantized_weight, _, _ = (
            NVFP4QTensor.quantize(data, BLOCK_SIZE, qscales_linear, scaling_factor_2, try_tensorrt=True))
        qweight = quantized_weight._quantized_data
        print(qweight.shape, qscales_linear.shape)
        qscales = _swizzle_scales(qscales_linear.unsqueeze(0), scale_ndim=3).squeeze(0)
        global_scale = 1.0 / scaling_factor_2
        return qweight, qscales, global_scale, qscales_linear


def _dequantize_linear_to_fp32(
    tensor_fp4, tensor_scales_linear, global_scale, dtype, device, block_size=16
):
    assert tensor_fp4.dtype == torch.uint8
    m, packed_k = tensor_fp4.shape
    k = packed_k * 2
    tensor_f32 = _break_fp4_bytes_v2(tensor_fp4)
    tensor_f32 = tensor_f32.reshape(m, k // block_size, block_size)
    block_scale = tensor_scales_linear.to(torch.float32) / global_scale

    # scale the tensor
    print(tensor_f32.shape, block_scale.shape)
    out = (tensor_f32 * block_scale.unsqueeze(-1)).reshape(m, k).to(device=device, dtype=dtype)
    return out

def recover_swizzled_scales(scale, m, k):
    rounded_m = ((m + 128 - 1) // 128) * 128
    scale_k = k // BLOCK_SIZE
    rounded_k = ((scale_k + 4 - 1) // 4) * 4
    # Recover the swizzled scaling factor to linear layout
    tmp = torch.reshape(scale, (1, rounded_m // 128, rounded_k // 4, 32, 4, 4))
    tmp = torch.permute(tmp, (0, 1, 4, 3, 2, 5))
    result = torch.reshape(tmp, (rounded_m, rounded_k)).to(torch.float32)
    return result[:m, :rounded_k]

@cuda_timeit
def _dequantize_swizzle_to_fp32(
    tensor_fp4, tensor_sf, global_scale, dtype, device, block_size=16
):
    assert tensor_fp4.dtype == torch.uint8
    m, packed_k = tensor_fp4.shape
    k = packed_k * 2
    tensor_f32 = _break_fp4_bytes_v2(tensor_fp4)
    tensor_f32 = tensor_f32.reshape(m, k // block_size, block_size)
    tensor_sf = tensor_sf.view(torch.float8_e4m3fn)
    tensor_sf_linear = recover_swizzled_scales(tensor_sf, m, k)
    block_scale = tensor_sf_linear.to(torch.float32) / global_scale

    # scale the tensor
    print(tensor_f32.shape, block_scale.shape)
    out = (tensor_f32 * block_scale.unsqueeze(-1)).reshape(m, k).to(device=device, dtype=dtype)
    return out

def print_array_bytes(arr, name, limit=5):
    if arr is None:
        print(f"{name} = None")
        return
    arr_cpu = arr.cpu().detach()
    bytes_data = arr_cpu.numpy().tobytes()
    total_bytes = len(bytes_data)
    print(f"{name} [{total_bytes} bytes]: ", end="")
    if total_bytes > 2 * limit:
        for i in range(limit):
            print(f"0x{bytes_data[i]:02x} ", end="")
        print("... ", end="")
        for i in range(total_bytes - limit, total_bytes):
            print(f"0x{bytes_data[i]:02x}" + (" " if i != total_bytes - 1 else ""), end="")
    else:
        for i in range(total_bytes):
            print(f"0x{bytes_data[i]:02x}" + (" " if i != total_bytes - 1 else ""), end="")
    print()

def print_array_values(arr, name, limit=5):
    if arr is None:
        print(f"{name} = None")
        return
    arr_cpu = arr.cpu().detach().flatten()
    total_elems = arr_cpu.numel()
    print(f"{name} [{total_elems}]: ", end="")
    if total_elems > 2 * limit:
        for i in range(limit):
            print(f"{arr_cpu[i].item():.6f} ", end="")
        print("... ", end="")
        for i in range(total_elems - limit, total_elems):
            print(f"{arr_cpu[i].item():.6f}" + (" " if i != total_elems - 1 else ""), end="")
    else:
        for i in range(total_elems):
            print(f"{arr_cpu[i].item():.6f}" + (" " if i != total_elems - 1 else ""), end="")
    print()

def print_nvfp4_matmul_info(a_fp4, a_scales, a_global_scale,
                             b_fp4, b_scales, b_global_scale, out):
    print("\n========== Python NVFP4 Matmul Debug Info ==========")
    print('m = {}, k = {}, n = {}'.format(a_fp4.shape[0], a_fp4.shape[1] * 2, b_fp4.shape[0]))
    a_gs = a_global_scale.item() if torch.is_tensor(a_global_scale) else a_global_scale
    b_gs = b_global_scale.item() if torch.is_tensor(b_global_scale) else b_global_scale
    print(f"a_global_scale: {a_gs}")
    print(f"b_global_scale: {b_gs}")
    print(f'alpha = {1.0 / (a_gs * b_gs)}')

    print_array_bytes(a_fp4, "a_fp4")
    print_array_bytes(b_fp4, "b_fp4")
    print_array_bytes(a_scales.view(torch.uint8), "a_scales")
    print_array_values(a_scales, "a_scales")
    print_array_bytes(b_scales.view(torch.uint8), "b_scales")
    print_array_values(b_scales, "b_scales")

    print_array_values(out, "out")
    print("=====================================================\n")


def nvfp4_matmul(a_fp4, a_scales, a_global_scale,
                 b_fp4, b_scales, b_global_scale,
                 dtype, device, block_size=16):
    _, m_k = a_fp4.shape
    _, n_k = b_fp4.shape
    assert m_k == n_k

    a = _dequantize_swizzle_to_fp32(a_fp4, a_scales, a_global_scale, dtype, device, block_size)
    b = _dequantize_swizzle_to_fp32(b_fp4, b_scales, b_global_scale, dtype, device, block_size)
    out = torch.matmul(a, b.t())

    print_nvfp4_matmul_info(a_fp4, a_scales, a_global_scale, 
                            b_fp4, b_scales, b_global_scale, out)
    return out


class Qwen3VLTextConfig:
    def __init__(self, hidden_size=2048, intermediate_size=6144, hidden_act="silu"):
        self.hidden_size = hidden_size
        self.intermediate_size = intermediate_size
        self.hidden_act = hidden_act


class Qwen3VLTextMLP_NVFP4(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.hidden_size = config.hidden_size
        self.intermediate_size = config.intermediate_size
        self.act_fn = nn.SiLU()

        self.register_buffer('gate_qweight', torch.empty(0))
        self.register_buffer('gate_qscales', torch.empty(0))
        self.register_buffer('gate_input_global_scale', torch.empty(0))
        self.register_buffer('gate_weight_global_scale', torch.empty(0))
        self.register_buffer('up_qweight', torch.empty(0))
        self.register_buffer('up_qscales', torch.empty(0))
        self.register_buffer('up_input_global_scale', torch.empty(0))
        self.register_buffer('up_weight_global_scale', torch.empty(0))
        self.register_buffer('down_qweight', torch.empty(0))
        self.register_buffer('down_qscales', torch.empty(0))
        self.register_buffer('down_input_global_scale', torch.empty(0))
        self.register_buffer('down_weight_global_scale', torch.empty(0))

    def forward(self, x_fp16):
        # x_fp16: [num_tokens, hidden_size]
        if len(x_fp16.shape) == 2:
            x_flat = x_fp16.unsqueeze(0)
        batch_size, seq, hidden = x_flat.shape
        total_tokens = seq * batch_size
        x_flat = x_flat.reshape(total_tokens, hidden)

        gate_q, gate_s, gate_gs, _ = _quantize_to_nvfp4(
            x_flat, preset_global_scale=self.gate_input_global_scale
        )
        # 2. gate投影 (NVFP4矩阵乘)
        gate_out_fp16 = nvfp4_matmul(
            gate_q, gate_s, gate_gs,
            self.gate_qweight, self.gate_qscales, self.gate_weight_global_scale,
            dtype=torch.float16, device=x_flat.device, block_size=BLOCK_SIZE
        )

        # 3. up投影
        up_q, up_s, up_gs, _ = _quantize_to_nvfp4(
            x_flat, preset_global_scale=self.up_input_global_scale
        )
        up_out_fp16 = nvfp4_matmul(
            up_q, up_s, up_gs,
            self.up_qweight, self.up_qscales, self.up_weight_global_scale,
            dtype=torch.float16, device=x_flat.device, block_size=BLOCK_SIZE
        )

        # 4. SiLU和Hadamard积（FP16）
        silu_out = self.act_fn(gate_out_fp16)
        hadamard_out = silu_out * up_out_fp16

        # 5. down投影（需要量化hadamard_out）
        had_q, had_s, had_gs, _ = _quantize_to_nvfp4(
            hadamard_out, preset_global_scale=self.down_input_global_scale)
        down_out_fp16 = nvfp4_matmul(
            had_q, had_s, had_gs,
            self.down_qweight, self.down_qscales, self.down_weight_global_scale,
            dtype=torch.float16, device=x_flat.device, block_size=BLOCK_SIZE
        )

        return down_out_fp16, gate_out_fp16, up_out_fp16, silu_out, hadamard_out

class Qwen3VLTextMoE_NVFP4(nn.Module):
    def __init__(self, config, num_experts=3):
        super().__init__()
        self.num_experts = num_experts
        self.hidden_size = config.hidden_size
        self.experts = nn.ModuleList([
            Qwen3VLTextMLP_NVFP4(config)
            for _ in range(num_experts)
        ])
        self.router = nn.Linear(self.hidden_size, num_experts, bias=True)

    def set_router_params(self, weight, bias):
        self.router.weight.data = weight
        self.router.bias.data = bias

    def set_expert_weights(self, expert_idx,
                           gate_qw, gate_qs, gate_gs,
                           up_qw, up_qs, up_gs,
                           down_qw, down_qs, down_gs):
        exp = self.experts[expert_idx]
        exp.gate_qweight = gate_qw
        exp.gate_qscales = gate_qs
        exp.gate_weight_global_scale = gate_gs
        exp.up_qweight = up_qw
        exp.up_qscales = up_qs
        exp.up_weight_global_scale = up_gs
        exp.down_qweight = down_qw
        exp.down_qscales = down_qs
        exp.down_weight_global_scale = down_gs

    def set_expert_input_gs(self, expert_idx, gate_input_gs, up_input_gs, down_input_gs):
        exp = self.experts[expert_idx]
        exp.gate_input_global_scale = gate_input_gs
        exp.up_input_global_scale = up_input_gs
        exp.down_input_global_scale = down_input_gs

    def forward(self, x_fp16):
        # router FP16
        logits = self.router(x_fp16)
        expert_idx = logits.argmax(dim=-1)  # [total_tokens]

        out = torch.zeros_like(x_fp16)
        intermediate = {
            'router_logits': logits,
            'expert_idx': expert_idx,
        }

        for i in range(self.num_experts):
            mask = (expert_idx == i)
            if mask.any():
                expert_in = x_fp16[mask]
                expert_out, gate_out, up_out, silu_out, had_out = self.experts[i](expert_in)
                out[mask] = expert_out
                intermediate[f'expert_{i}'] = {
                    'output': expert_out,
                    'gate_out': gate_out,
                    'up_out': up_out,
                    'silu_out': silu_out,
                    'hadamard_out': had_out
                }

        return out, intermediate

def load_moe_nvfp4_weights(model, fp16_weight_file, save_path=None, moe_layer_idx=0):
    if fp16_weight_file is None:
        if save_path is not None and os.path.exists(save_path):
            moe_weights = load_file(save_path)
            model.set_router_params(
                moe_weights[f'model.layers.{moe_layer_idx}.mlp.router_weight'].transpose(1, 0).contiguous(), 
                moe_weights[f'model.layers.{moe_layer_idx}.mlp.router_bias']
            )
            for i in range(model.num_experts):
                model.set_expert_weights(
                    i,
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_qweight'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_qscales'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_input_global_scale'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_weight_global_scale'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_qweight'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_qscales'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_input_global_scale'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_weight_global_scale'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_qweight'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_qscales'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_input_global_scale'][i],
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_weight_global_scale'][i]
                )
        else:
            raise FileNotFoundError("No valid weight file found.")
    else:
        fp16_weights = load_file(fp16_weight_file)
        router_weight = fp16_weights[f'model.layers.{moe_layer_idx}.mlp.router_weight']
        router_bias = fp16_weights[f'model.layers.{moe_layer_idx}.mlp.router_bias']
        gate_weights = fp16_weights[f'model.layers.{moe_layer_idx}.mlp.experts_gate_proj_weight']
        up_weights = fp16_weights[f'model.layers.{moe_layer_idx}.mlp.experts_up_proj_weight']
        down_weights = fp16_weights[f'model.layers.{moe_layer_idx}.mlp.experts_down_proj_weight']

        model.set_router_params(router_weight.transpose(1, 0).contiguous(), router_bias)
        moe_weights = {}
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.router_weight'] = router_weight.cpu()
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.router_bias'] = router_bias.cpu()
        gate_qw_list = []
        gate_qs_list = []
        gate_gs_list = []
        up_qw_list = []
        up_qs_list = []
        up_gs_list = []
        down_qw_list = []
        down_qs_list = []
        down_gs_list = []
        for i in range(model.num_experts):
            gate_qw, gate_qs, gate_gs, _ = _quantize_to_nvfp4(gate_weights[i].transpose(1, 0).contiguous())
            up_qw, up_qs, up_gs, _ = _quantize_to_nvfp4(up_weights[i].transpose(1, 0).contiguous())
            down_qw, down_qs, down_gs, _ = _quantize_to_nvfp4(down_weights[i].transpose(1, 0).contiguous())
            model.set_expert_weights(i, gate_qw, gate_qs, gate_gs,
                                        up_qw, up_qs, up_gs,
                                        down_qw, down_qs, down_gs)
            gate_qw_list.append(gate_qw)
            gate_qs_list.append(gate_qs)
            gate_gs_list.append(gate_gs)
            up_qw_list.append(up_qw)
            up_qs_list.append(up_qs)
            up_gs_list.append(up_gs)
            down_qw_list.append(down_qw)
            down_qs_list.append(down_qs)
            down_gs_list.append(down_gs)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_qweight'] = torch.stack(gate_qw_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_qscales'] = torch.stack(gate_qs_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_input_global_scale'] = torch.tensor(
            [701.0, 363.0, 363.0], dtype=torch.float32) # 363.0
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_weight_global_scale'] = torch.stack(gate_gs_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_qweight'] = torch.stack(up_qw_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_qscales'] = torch.stack(up_qs_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_input_global_scale'] = torch.tensor(
            [701.0, 363.0, 363.0], dtype=torch.float32)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_weight_global_scale'] = torch.stack(up_gs_list, dim=0)

        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_qweight'] = torch.stack(down_qw_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_qscales'] = torch.stack(down_qs_list, dim=0)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_input_global_scale'] = torch.tensor(
            [146.0, 201.25, 209.875], dtype=torch.float32) # 205.25
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_weight_global_scale'] = torch.stack(down_gs_list, dim=0)

        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_alpha'] = torch.ones(model.num_experts, dtype=torch.float32)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_alpha'] = torch.ones(model.num_experts, dtype=torch.float32)
        moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_alpha'] = torch.ones(model.num_experts, dtype=torch.float32)
        for i in range(model.num_experts):
            moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_alpha'][i] = float(1.0 / (
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_input_global_scale'][i] *
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_weight_global_scale'][i]))
            moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_alpha'][i] = float(1.0 / (
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_input_global_scale'][i] *
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_weight_global_scale'][i]))
            moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_alpha'][i] = float(1.0 / (
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_input_global_scale'][i] *
                    moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_weight_global_scale'][i]))
        for i in range(model.num_experts):
            model.set_expert_input_gs(i, 
                moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.gate_input_global_scale'][i],
                moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.up_input_global_scale'][i],
                moe_weights[f'model.layers.{moe_layer_idx}.mlp.experts.down_input_global_scale'][i]
            )

        if save_path is not None and not os.path.exists(save_path):
            print(f"✅ 量化权重已保存到 {save_path}.")
            save_file(moe_weights, save_path)
    return model

def generate_input_nvfp4(filename, seq_len=1, dtype=torch.float16):
    if not os.path.exists(filename):
        torch.manual_seed(42)
        np.random.seed(42)
        
        batch_size = 1
        hidden_size = 2048
        input = torch.randn(batch_size, seq_len, hidden_size).to(dtype)
        inputs = {}
        inputs['hidden_state'] = input

        save_file(inputs, filename)
        print(f"✅ 测试输入已保存到 {filename}, 输入形状: {input.shape}.")
    else:
        input = load_file(filename)['hidden_state']
        print(f"✅ 测试输入已从 {filename} 加载, 输入形状: {input.shape}.")
    return input


def print_data_info_python(model, x, intermediate_results, output):
    print("=" * 80)
    print("PYTHON IMPLEMENTATION DATA INFO")
    print("=" * 80)
    
    # 基本参数
    batch_size = x.shape[0]
    seq_len = x.shape[1]
    hidden_size = model.hidden_size
    intermediate_size = model.experts[0].intermediate_size
    experts_num = model.num_experts
    experts_topk = 1
    total_tokens = batch_size * seq_len
    
    print(f"MoE Input Parameters:")
    print(f"  Batch Size: {batch_size}")
    print(f"  Sequence Length: {seq_len}")
    print(f"  Hidden Size: {hidden_size}")
    print(f"  Intermediate Size: {intermediate_size}")
    print(f"  Experts Number: {experts_num}")
    print(f"  Experts Top-K: {experts_topk}")
    print(f"  Total Tokens: {total_tokens}")
    
    # 打印路由器权重
    router_weight = model.router.weight.data
    router_bias = model.router.bias.data
    print(f"\nRouter weight shape: {router_weight.shape}, dtype: {router_weight.dtype}")
    print(f"Router weight data (first 5 elements):")
    router_weight_flat = router_weight.flatten()
    for i in range(min(5, len(router_weight_flat))):
        print(f"  [{i}] = {router_weight_flat[i].item():.6f}")
    
    print(f"\nRouter bias data:")
    for i in range(len(router_bias)):
        print(f"  [{i}] = {router_bias[i].item():.6f}")
    
    for expert_idx in range(experts_num):
        print(f"\nExpert {expert_idx}:")
        print(f"\nExpert {expert_idx} gate_proj qweight (first 5 elements):")
        gate_qweight = model.experts[expert_idx].gate_qweight.flatten()
        for i in range(min(5, len(gate_qweight))):
            print(f"  [{i}] = {gate_qweight[i].item()}")
        print(f"\nExpert {expert_idx} gate_proj qscales (first 5 elements):")
        gate_qscales = model.experts[expert_idx].gate_qscales.flatten()
        for i in range(min(5, len(gate_qscales))):
            print(f"  [{i}] = {gate_qscales[i].view(torch.uint8).item()}")

        print(f"\nExpert {expert_idx} up_proj qweight (first 5 elements):")
        up_qweight = model.experts[expert_idx].up_qweight.flatten()
        for i in range(min(5, len(up_qweight))):
            print(f"  [{i}] = {up_qweight[i].item()}")
        print(f"\nExpert {expert_idx} up_proj qscales (first 5 elements):")
        up_qscales = model.experts[expert_idx].up_qscales.flatten()
        for i in range(min(5, len(up_qscales))):
            print(f"  [{i}] = {up_qscales[i].view(torch.uint8).item()}")

        print(f"\nExpert {expert_idx} down_proj qweight (first 5 elements):")
        down_qweight = model.experts[expert_idx].down_qweight.flatten()
        for i in range(min(5, len(down_qweight))):
            print(f"  [{i}] = {down_qweight[i].item()}")
        print(f"\nExpert {expert_idx} down_proj qscales (first 5 elements):")
        down_qscales = model.experts[expert_idx].down_qscales.flatten()
        for i in range(min(5, len(down_qscales))):
            print(f"  [{i}] = {down_qscales[i].view(torch.uint8).item()}")

    print(f"\nRouter logits:")
    logits = intermediate_results['router_logits']
    print(f"  Shape: {logits.shape}, dtype: {logits.dtype}")
    for i in range(logits.shape[-1]):
        print(f"  Expert {i}: {logits[0, 0, i].item():.6f}")
    
    print(f"\nSelected expert indices:")
    expert_idx = intermediate_results['expert_idx']
    for i in range(total_tokens):
        selected_expert = expert_idx[0, i].item()
        print(f"  Token {i} -> Expert {selected_expert}")

        print(f"\nInput data (first 5 elements):")
        input_data = x[0, i]
        for j in range(min(5, len(input_data))):
            print(f"  [{j}] = {input_data[j].item():.6f}")

        # 打印中间结果（如果有）
        if f'expert_{selected_expert}' in intermediate_results:
            expert_result = intermediate_results[f'expert_{selected_expert}']
            
            print(f"\nExpert {selected_expert} intermediate results:")
            
            if 'gate_out' in expert_result:
                gate_out = expert_result['gate_out']
                print(f"  Gate output shape: {gate_out.shape}, dtype: {gate_out.dtype}")
                print(f"  Gate output (first 5 elements):")
                gate_flat = gate_out.flatten()
                for i in range(min(5, len(gate_flat))):
                    print(f"    [{i}] = {gate_flat[i].item():.6f}")
            
            if 'up_out' in expert_result:
                up_out = expert_result['up_out']
                print(f"  Up output shape: {up_out.shape}, dtype: {up_out.dtype}")
                print(f"  Up output (first 5 elements):")
                up_flat = up_out.flatten()
                for i in range(min(5, len(up_flat))):
                    print(f"    [{i}] = {up_flat[i].item():.6f}")
            
            if 'silu_out' in expert_result:
                silu_out = expert_result['silu_out']
                print(f"  SiLU output shape: {silu_out.shape}, dtype: {silu_out.dtype}")
                print(f"  SiLU output (first 5 elements):")
                silu_flat = silu_out.flatten()
                for i in range(min(5, len(silu_flat))):
                    print(f"    [{i}] = {silu_flat[i].item():.6f}")
            
            if 'hadamard_out' in expert_result:
                hadamard_out = expert_result['hadamard_out']
                print(f"  Hadamard output shape: {hadamard_out.shape}, dtype: {hadamard_out.dtype}")
                print(f"  Hadamard output (first 5 elements):")
                hadamard_flat = hadamard_out.flatten()
                for i in range(min(5, len(hadamard_flat))):
                    print(f"    [{i}] = {hadamard_flat[i].item():.6f}")
            
            if 'output' in expert_result:
                expert_output = expert_result['output']
                print(f"  Expert output shape: {expert_output.shape}, dtype: {expert_output.dtype}")
                print(f"  Expert output (first 5 elements):")
                output_flat = expert_output.flatten()
                for i in range(min(5, len(output_flat))):
                    print(f"    [{i}] = {output_flat[i].item():.6f}")

    print(f"\nFinal output shape: {output.shape}, dtype: {output.dtype}")
    print(f"Final output data (first 10 elements):")
    output_flat = output.flatten()
    for i in range(min(10, len(output_flat))):
        print(f"  [{i}] = {output_flat[i].item():.6f}")


def save_intermediate_results(intermediate_results, output_file, moe_layer_idx=0, experts_num=3):
    results = {}
    results['expert_idx'] = intermediate_results['expert_idx']
    for expert_idx in range(experts_num):
        if f'expert_{expert_idx}' in intermediate_results:
            results[f'model.layers.{moe_layer_idx}.mlp.experts.{expert_idx}.gate_out'] = intermediate_results[f'expert_{expert_idx}']['gate_out']
            results[f'model.layers.{moe_layer_idx}.mlp.experts.{expert_idx}.up_out'] = intermediate_results[f'expert_{expert_idx}']['up_out']
            results[f'model.layers.{moe_layer_idx}.mlp.experts.{expert_idx}.silu_out'] = intermediate_results[f'expert_{expert_idx}']['silu_out']
            results[f'model.layers.{moe_layer_idx}.mlp.experts.{expert_idx}.hadamard_out'] = intermediate_results[f'expert_{expert_idx}']['hadamard_out']
            results[f'model.layers.{moe_layer_idx}.mlp.experts.{expert_idx}.output'] = intermediate_results[f'expert_{expert_idx}']['output']
    save_file(results, output_file)
    print(f"✅ 中间结果已保存到 {output_file}")


def generate_nvfp4_test_data(cpu_offload=True):
    FP16_WEIGHT_FILE = os.path.join(DATA_ROOT_DIR, "moe_weights.safetensors")
    FP16_INPUT_FILE = os.path.join(DATA_ROOT_DIR, "seq1/moe_input.safetensors")
    OUTPUT_DIR = os.path.join(DATA_ROOT_DIR, "seq1")
    os.makedirs(OUTPUT_DIR, exist_ok=True)

    NVFP4_WEIGHT_FILE = os.path.join(OUTPUT_DIR, "moe_nvfp4_weights.safetensors")
    NVFP4_OUTPUT_FILE = os.path.join(OUTPUT_DIR, "moe_nvfp4_output.safetensors")
    NVFP4_INTERMEDIATE_FILE = os.path.join(OUTPUT_DIR, "moe_nvfp4_intermediate_results.safetensors")

    experts_num = 3
    config = Qwen3VLTextConfig(hidden_size=2048, intermediate_size=6144)
    model_nvfp4 = Qwen3VLTextMoE_NVFP4(config, num_experts=experts_num)
    model = load_moe_nvfp4_weights(model_nvfp4, FP16_WEIGHT_FILE, NVFP4_WEIGHT_FILE)

    if cpu_offload:
        model = model.eval().cpu()
    else:
        model = model.eval().cuda()
    model.eval()

    print("运行NVFP4模拟前向...")
    with torch.no_grad():
        x = generate_input_nvfp4(FP16_INPUT_FILE)
        if not cpu_offload:
            x = x.cuda()
        output, intermediate_results = model_nvfp4(x)
        print_data_info_python(model, x, intermediate_results, output)

    print("保存文件...")
    save_file({'output': output.cpu()}, NVFP4_OUTPUT_FILE)
    save_intermediate_results(intermediate_results, NVFP4_INTERMEDIATE_FILE, experts_num=experts_num)

    print("完成！生成的文件：")
    print(f"  {NVFP4_WEIGHT_FILE}")
    print(f"  {NVFP4_OUTPUT_FILE}")
    print(f"  {NVFP4_INTERMEDIATE_FILE}")


def compare_output():
    output_file = os.path.join(DATA_ROOT_DIR, "seq1/moe_nvfp4_output.safetensors")
    ref_output_file = os.path.join(DATA_ROOT_DIR, "seq1/moe_output_ref.safetensors")
    output = load_file(output_file)['output']
    ref_output = load_file(ref_output_file)['output']

    diff = torch.abs(output - ref_output)
    max_diff = diff.max().item()
    mean_diff = diff.mean().item()
    
    print(f"最大绝对误差: {max_diff:.10f}")
    print(f"平均绝对误差: {mean_diff:.10f}")

    if torch.allclose(output, ref_output, rtol=1e-5, atol=1e-5):
        print("✅ 输出一致!")
        return True
    else:
        print("❌ 输出不一致!")
        print(output)
        print(ref_output)
        return False


generate_nvfp4_test_data(False)
# compare_output()